<a href="https://colab.research.google.com/github/SHRAVAN-AMBEER/Deep_Learning_Practice/blob/main/DL_week4(168).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Q14:Implement the MLP using the Types of GD (BGD,SGD,Mini BatchGD, SGD with Momentum, SGD with Nesterov,Adagrad, RMSProp,Adadelta and Adam) for learning XOR operation

Batch GD (Size=4): Too slow. It only updates weights once per epoch and fails to converge within 500 steps.

Stochastic GD (Size=1): Fast but erratic. It updates frequently, but the path to the minimum is noisy and jagged.

Mini-Batch GD (Size=2): The sweet spot for standard SGD, balancing update speed with gradient stability.

Momentum: A major upgrade for SGD. It builds velocity to roll through flat regions, resulting in a much lower final loss.

Nesterov Momentum: Adds precision to Momentum by slowing down right before the minimum, preventing the model from overshooting.

Adagrad: Fails on longer runs. The learning rate decays too aggressively, freezing the model before it can finish learning.

RMSprop: Fixes Adagrad's freezing issue. It maintains a healthy learning rate and solves XOR quickly and reliably.

Adam: The clear winner. It combines the speed of Momentum with the adaptive learning of RMSprop to consistently hit 100% accuracy with the lowest possible loss.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD, Adagrad, RMSprop, Adadelta, Adam

# XOR Dataset
X = np.array([[0,0], [0,1], [1,0], [1,1]], dtype=np.float32)
y = np.array([[0], [1], [1], [0]], dtype=np.float32)

def train_model(optimizer, batch_size, name):
    # 1. Reset seed INSIDE the function so every optimizer starts from the exact same weights
    tf.random.set_seed(42)
    np.random.seed(42)

    # 2. Build the Model
    model = Sequential([
        Dense(8, activation='tanh', input_shape=(2,)),
        Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer=optimizer,
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    # 3. Train the Model
    history = model.fit(X, y, epochs=500, batch_size=batch_size, verbose=0)

    # 4. Extract and print results
    final_loss = history.history['loss'][-1]
    final_acc = history.history['accuracy'][-1]

    print(f"{name:<20} | Batch: {batch_size} | Loss: {final_loss:.4f} | Accuracy: {final_acc:.2f}")

# --- Run the Experiment ---
print("-" * 60)
print(f"{'Optimizer Strategy':<20} | {'Batch'} | {'Final Loss'} | {'Accuracy'}")
print("-" * 60)

# SGD Variations
train_model(SGD(learning_rate=0.5), 4, "Batch GD (LR=0.5)")
train_model(SGD(learning_rate=0.1), 1, "Stochastic GD")
train_model(SGD(learning_rate=0.1), 2, "Mini-Batch GD")

# Momentum Variations
train_model(SGD(learning_rate=0.1, momentum=0.9), 2, "SGD + Momentum")
train_model(SGD(learning_rate=0.1, momentum=0.9, nesterov=True), 2, "Nesterov Momentum")

# Adaptive Optimizers
train_model(Adagrad(learning_rate=0.1), 2, "Adagrad")
train_model(RMSprop(learning_rate=0.01), 2, "RMSProp")
# Note: Keras Adadelta default LR is 0.001 which is too slow. 1.0 is the standard paper value.
train_model(Adadelta(learning_rate=1.0), 2, "Adadelta")
train_model(Adam(learning_rate=0.01), 2, "Adam")
print("-" * 60)

------------------------------------------------------------
Optimizer Strategy   | Batch | Final Loss | Accuracy
------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Batch GD (LR=0.5)    | Batch: 4 | Loss: 0.0076 | Accuracy: 1.00
Stochastic GD        | Batch: 1 | Loss: 0.0135 | Accuracy: 1.00
Mini-Batch GD        | Batch: 2 | Loss: 0.0976 | Accuracy: 1.00
SGD + Momentum       | Batch: 2 | Loss: 0.0018 | Accuracy: 1.00
Nesterov Momentum    | Batch: 2 | Loss: 0.0018 | Accuracy: 1.00
Adagrad              | Batch: 2 | Loss: 0.0689 | Accuracy: 1.00
RMSProp              | Batch: 2 | Loss: 0.0000 | Accuracy: 1.00
Adadelta             | Batch: 2 | Loss: 0.3395 | Accuracy: 1.00
Adam                 | Batch: 2 | Loss: 0.0043 | Accuracy: 1.00
------------------------------------------------------------


#ON MY CHOOSEN DATASET(fashion_mnist)

Because Fashion MNIST has 70,000 images (instead of just 4 rows of XOR data), I made two necessary optimizations:

Batch Size: Increased to 64. Using a batch size of 1 or 2 on 70,000 images would take hours to compute.

Epochs: Reduced to 10. Deep learning models on this dataset show their trajectory very clearly within 10 epochs, saving you massive amounts of computation time.

When classifying complex image data like Fashion MNIST, here is how the optimizers behave:

Standard SGD: The slowest learner. It takes small, rigid steps and needs significantly more than 10 epochs to achieve a high accuracy on complex image features.

Momentum & Nesterov: A massive improvement over basic SGD. By building "velocity" across the image dataset gradients, it learns the clothing features much faster and hits a high accuracy quickly.

Adagrad: Starts strong but stalls out. Because it constantly shrinks the learning rate, it often stops learning before it fully masters the complex pixel patterns.

RMSprop: Excellent performance. It successfully navigates the image data by adapting the learning rate without letting it decay to zero, resulting in high test accuracy.

Adam: The overall winner. By perfectly combining the speed of Momentum with the adaptive, per-parameter learning rates of RMSprop, Adam achieves the highest test accuracy and the lowest loss in the shortest amount of time.

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import SGD, Adagrad, RMSprop, Adam

# 1. Load and Preprocess Fashion MNIST
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Normalize pixel values to [0, 1] for stable gradients
X_train, X_test = X_train / 255.0, X_test / 255.0

def train_model(optimizer, name):
    # Fair starting line: Every model gets the exact same initial random weights
    tf.random.set_seed(42)

    # Build a simple MLP for images
    model = Sequential([
        Flatten(input_shape=(28, 28)),     # Flatten 28x28 images to a 1D array
        Dense(128, activation='relu'),     # Hidden layer
        Dense(10, activation='softmax')    # 10 output classes for clothing types
    ])

    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    # Train the Model (Batch size 64 for speed on large datasets)
    model.fit(X_train, y_train, epochs=10, batch_size=64, verbose=0)

    # Evaluate on the unseen Test Data
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    print(f"{name:<20} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

# --- Run the Experiment ---
print("-" * 55)
print(f"{'Optimizer Strategy':<20} | {'Test Loss'}  | {'Test Acc'}")
print("-" * 55)

# Note: Using standard default learning rates for a fair real-world comparison
train_model(SGD(learning_rate=0.01), "Standard SGD")
train_model(SGD(learning_rate=0.01, momentum=0.9), "SGD + Momentum")
train_model(SGD(learning_rate=0.01, momentum=0.9, nesterov=True), "Nesterov Momentum")
train_model(Adagrad(learning_rate=0.01), "Adagrad")
train_model(RMSprop(learning_rate=0.001), "RMSProp")
train_model(Adam(learning_rate=0.001), "Adam")
print("-" * 55)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
-------------------------------------------------------
Optimizer Strategy   | Test Loss  | Test Acc
-------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Standard SGD         | Test Loss: 0.4723 | Test Acc: 0.8318
SGD + Momentum       | Test Loss: 0.3493 | Test Acc: 0.8754
Nesterov Momentum    | Test Loss: 0.3577 | Test Acc: 0.8721
Adagrad              | Test Loss: 0.4201 | Test Acc: 0.8505
RMSProp              | Test Loss: 0.4404 | Test Acc: 0.8596
Adam                 | Test Loss: 0.3478 | Test Acc: 0.8771
-------------------------------------------------------
